***5D) AGE x SEVERITY INTERACTION (does the SLEDAI relationship differ between child and adult?)***

In [ ]:
#SET BASE DIRECTORY
#This notebook expects the GSE135779 data folders (adult_individual_h5ad, GSE135779_RAW, Results, etc.) to sit one level above this Notebooks folder. Update BASE_DIR below if your data lives elsewhere.

import os
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), ".."))

In [ ]:
from pathlib import Path
import sys

sys.path.insert(0, str(Path(BASE_DIR) / "Notebooks"))
from publication_utils import configure_publication_notebook

FIGURE_DIR = configure_publication_notebook(BASE_DIR, "06B_GSE135779_AGE_SEVERITY_INTERACTION")


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

metadata = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/patient_clinical_metadata.csv")

child_scores = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/cSLE_per_patient_scores.csv")
child_scores["age_group"] = "pediatric"

adult_scores = pd.read_csv(f"{BASE_DIR}/Results/severity_analysis/aSLE_per_patient_scores.csv")
adult_scores["age_group"] = "adult"

combined = pd.concat([child_scores, adult_scores], ignore_index=True)
combined = combined.merge(metadata[["sample", "SLEDAI"]], on="sample", how="left")
combined = combined.dropna(subset=["SLEDAI", "score"])

# only test interactions present in both cohorts
shared_interactions = set(child_scores["interaction_id"]) & set(adult_scores["interaction_id"])
print(f"Interactions shared between child and adult: {len(shared_interactions)}")

combined = combined[combined["interaction_id"].isin(shared_interactions)]
print(f"Rows available for testing: {len(combined)}")

In [ ]:
records = []
model_failures = []
from analysis_config import MIN_PATIENTS_PER_GROUP
min_per_group = MIN_PATIENTS_PER_GROUP

for interaction_id, group in combined.groupby("interaction_id"):
    n_child = group.loc[group["age_group"] == "pediatric", "sample"].nunique()
    n_adult = group.loc[group["age_group"] == "adult", "sample"].nunique()
    if n_child < min_per_group or n_adult < min_per_group:
        continue
    if group["score"].std() == 0:
        continue

    try:
        model = smf.ols("score ~ SLEDAI * C(age_group, Treatment(reference=\"adult\"))", data=group).fit(cov_type="HC3")
        interaction_term = [c for c in model.pvalues.index if ":" in c and "SLEDAI" in c]
        if not interaction_term:
            continue
        term = interaction_term[0]
        p_interaction = model.pvalues[term]
        coef_interaction = model.params[term]
        ci_low, ci_high = model.conf_int().loc[term]
    except Exception as exc:
        model_failures.append({"interaction_id": interaction_id, "error_type": type(exc).__name__, "message": str(exc)})
        continue

    row = group.iloc[0]
    records.append({
        "interaction_id": interaction_id,
        "source": row["source"], "target": row["target"],
        "ligand_complex": row["ligand_complex"], "receptor_complex": row["receptor_complex"],
        "n_child": n_child, "n_adult": n_adult,
        "interaction_coef": coef_interaction,
        "interaction_ci95_low": ci_low,
        "interaction_ci95_high": ci_high,
        "n_observations": int(model.nobs),
        "r_squared": model.rsquared,
        "condition_number": model.condition_number,
        "interaction_p_value": p_interaction,
    })

failure_columns = ["interaction_id", "error_type", "message"]
pd.DataFrame(model_failures, columns=failure_columns).to_csv(f"{BASE_DIR}/Results/severity_analysis/age_severity_model_failures.csv", index=False)

result = pd.DataFrame(records)
if len(result) > 0:
    result["fdr_q_value"] = multipletests(result["interaction_p_value"], method="fdr_bh")[1]
    result = result.sort_values("fdr_q_value")

out_path = f"{BASE_DIR}/Results/severity_analysis/age_severity_interaction.csv"
result.to_csv(out_path, index=False)

print(f"Tested {len(result)} interactions for an age x SLEDAI interaction effect")
print(f"Significant at FDR < 0.05: {(result['fdr_q_value'] < 0.05).sum() if len(result) else 0}")
print(f"Significant at FDR < 0.10: {(result['fdr_q_value'] < 0.10).sum() if len(result) else 0}")
print(f"\nSaved: {out_path}")
result.head(20)

**Age x severity interaction volcano plot.**

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7, 5))
sig = result["fdr_q_value"] < 0.05
ax.scatter(result.loc[~sig, "interaction_coef"], -np.log10(result.loc[~sig, "interaction_p_value"]),
           s=8, alpha=0.3, color="gray", label="not significant")
ax.scatter(result.loc[sig, "interaction_coef"], -np.log10(result.loc[sig, "interaction_p_value"]),
           s=10, alpha=0.8, color="#A6392A", label="FDR < 0.05")
ax.set_xlabel("Interaction coefficient (age_group x SLEDAI)")
ax.set_ylabel("-log10(p-value)")
ax.set_title(f"{sig.sum()}/{len(result)} interactions with significant age x severity interaction")
ax.legend(fontsize=8)
plt.tight_layout()
out_path = f"{BASE_DIR}/Results/severity_analysis/age_severity_interaction_volcano.png"
plt.savefig(out_path, dpi=600)
plt.show()
print("Saved:", out_path)